### Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:
1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

##### Overview
A typical RAG evaluation workflow consists of three main steps:
1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
* Answer relevance
* Answer accuracy
* Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

#### Chatbot Evaluation

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [4]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['d0cf7dfc-a240-46f2-8f80-679ca4c8bab4',
  'd0225e60-705f-4385-9565-3c45b106f9bd',
  'b8f4a2a2-cacd-4a1c-b40c-90c46a9c38a6',
  'ff03f443-e4cf-4ba9-bba8-f7413344b880',
  'bc81d75d-f738-4564-94bd-d76ba2969547'],
 'count': 5,
 'as_of': '2026-10-01T03:59:06.581868607Z'}

#### Define Metrics - LLM as a Judge

In [8]:
from groq import Groq
from langsmith import wrappers
 
groq_client = Groq()

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict) -> bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response = groq_client.chat.completions.create(
            model = "openai/gpt-oss-120b",
            temperature = 0,
            messages = [
                  {"role": "system", "content": eval_instructions},
                  {"role": "user", "content": user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"

In [15]:
# Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

#### Run Evaluations

In [16]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "openai/gpt-oss-120b", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model = model,
        temperature = 0,
        messages = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [17]:
# Call my_app for every datapoints

def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [18]:
# Run our evaluation
experiment_results = client.evaluate(
    ls_target, # Your AI system
    data = dataset_name,
    evaluators = [correctness,concision],
    experiment_prefix = "openai/gpt-oss-120b"
)

View the evaluation results for experiment: 'openai/gpt-oss-120b-9c8cc53e' at:
https://smith.langchain.com/o/a4b4c27f-a7c6-46b4-af97-3acf8ca7bc5b/datasets/1ab46d8d-be0a-439f-96fe-68360352890b/compare?selectedSessions=8f9aafe7-827d-45c8-acc2-1c31953d4977




5it [00:22,  4.59s/it]


In [19]:
# Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model = "openai/gpt-oss-120b")}

In [20]:
# Run our evaluation

experiment_results = client.evaluate(
    ls_target, # Your AI system
    data = dataset_name,
    evaluators = [correctness,concision],
    experiment_prefix = "openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-fc168557' at:
https://smith.langchain.com/o/a4b4c27f-a7c6-46b4-af97-3acf8ca7bc5b/datasets/1ab46d8d-be0a-439f-96fe-68360352890b/compare?selectedSessions=01346ba3-ec63-42e7-83d4-2ef59356e0c6




5it [00:08,  1.61s/it]


#### Evaluation for RAG

In [24]:
# RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the "vector store" using OpenAIEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embedding = embeddings,
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4775.74it/s]


In [25]:
retriever.invoke("what are Agents?")

[Document(id='708dfae8-b715-4d55-a942-d12350a9febc', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [26]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D9A338D810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D9A32DE120>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
from langsmith import traceable

# Add decorator
@traceable()
def rag_bot(question:str)->dict:
    # Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    # llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer": ai_msg.content, "documents": docs}

In [29]:
rag_bot("What are Agents?")

{'answer': 'Agents are autonomous systems whose core “brain” is a large language model (LLM) that interacts with the world via planning, memory, and tool‑use components. They decompose tasks into subgoals, reflect on past actions, and store experiences in long‑term memory to guide future behavior. This architecture lets them act and collaborate like virtual characters or problem‑solving assistants.',
 'documents': [Document(id='708dfae8-b715-4d55-a942-d12350a9febc', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-p

#### Dataset

In [30]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

# Create the daatset and example in Langsmith
dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name = dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

{'example_ids': ['e4589313-8a83-4e67-b387-3ff502f3f1ae',
  '3c97a1a3-c198-4cb5-b88e-c8ba88476ec0',
  '145f253a-fc61-4fbd-976f-4336f0d4acdb'],
 'count': 3,
 'as_of': '2026-10-01T08:05:35.762319925Z'}

#### Evaluators or Metrics
Correctness: Response vs reference answer
* Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
* Mode: Requires a ground truth (reference) answer supplied through a dataset
* Evaluator: Use LLM-as-judge to assess answer correctness.

In [32]:
from typing_extensions import Annotated,TypedDict

# Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(CorrectnessGrade, method="json_schema", strict=True)

# Evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

#### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance - as in, did the model address the user's question or not.

In [33]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

#### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [34]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

#### Retrieval Relevance: Retrieved docs vs input

In [35]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

#### Run the evaluation

In [37]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data = dataset_name,
    evaluators = [correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix = "rag-doc-relevance",
    metadata = {"version": "LCEL context, gpt-4-0125-preview"},
)

# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-f0495c73' at:
https://smith.langchain.com/o/a4b4c27f-a7c6-46b4-af97-3acf8ca7bc5b/datasets/3cdfc3e7-61ed-42d6-93ce-0a1925707331/compare?selectedSessions=6d5e0365-f758-4b2b-bb76-9c1f8227d5aa




3it [01:54, 38.27s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are five types of adversarial attacks?,The five types of adversarial attacks are Toke...,[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,1.111599,145f253a-fc61-4fbd-976f-4336f0d4acdb,01a0f683-41a4-7e41-ba2d-4813660d0915
1,What are the types of biases that can arise wi...,Few‑shot prompting can suffer from several sys...,[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,True,True,True,8.585255,3c97a1a3-c198-4cb5-b88e-c8ba88476ec0,01a0f683-816a-76b0-aec0-b6baca3b5ede
2,How does the ReAct agent use self-reflection?,The ReAct agent itself follows a Thought → Act...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",False,True,True,True,13.168772,e4589313-8a83-4e67-b387-3ff502f3f1ae,01a0f684-34a8-7870-8f07-5a0142ef2c8d
